# 3. Rezultati

Šta se ovde pokazuje: koliko je naučena politika dobra na sintetici, prenosi li
se na instance iz literature i na stvaran grad, i šta od zamišljenog nije radilo.

> **Napomena:** tabele u `results/` nisu u repou nego ih prave skripte iz
> `tndp/experiments/` (vidi README). Brojevi ispod su sa modela `gravity-v2h` za
> sintetiku i `novisad-r19h` za Novi Sad.

In [1]:
import sys
from pathlib import Path

KOREN = Path.cwd().parent
sys.path.insert(0, str(KOREN))

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [2]:
from IPython.display import Markdown, Image

def tabela(ime):
    return Markdown((KOREN / "results" / f"{ime}.md").read_text(encoding="utf-8"))

## 3.1 Held-out sintetika

Znači da je naučena heuristika bolja od konstruktivne, ali da ne zamenjuje pretragu.

Poređenje po broju evaluacija to i traži: lokalna pretraga troši 3000 evaluacija
cilja po gradu, plus oko 850 na greedy start iz kog kreće, a sampling 32 troši 32. Pod istim vremenskim budžetom odnos se menja
(`anytime.md`), a najbolji rezultat u repou ne daje nijedna metoda sama nego njihov
spoj: lokalna pretraga koja kreće iz mreže koju je dala politika (`hybrid.md`).
Uz to, `alpha = 0.5` je za politiku nepovoljna tačka fronta; iznad `alpha ~ 0.59`
prestiže i lokalnu pretragu (`pareto.md`).

In [3]:
tabela('main-20-v2h')

# Held-out sintetika (20 gradova, n [15, 30], R=4, alpha=0.5, model runs/gravity-v2h/best.pt)

`cilj` = alpha * C_p_all/donja_granica + (1-alpha) * C_o/MST; manje je bolje.
Nepokrivena tražnja je već u C_p_all (nepokriven par se naplaćuje 8x uličnim
najkraćim vremenom), pa nema zasebne kazne. `C_p` je prosek samo nad opsluženim
parovima i dat je radi poređenja sa literaturom, između metoda sa različitim
`d_un` nije uporediv, za to služi `C_p_all`.

± je standardna devijacija po gradovima. Δ i p su **uparene** razlike u `cilj`
u odnosu na `greedy` (Wilcoxon, isti gradovi); Δ>0 znači bolje od reference.

| metoda | cilj | Δ vs greedy | p | C_p_all | C_p | C_o | d_0 | d_un | s/grad |
|---|---|---|---|---|---|---|---|---|---|
| random 200 | 2.149 ± 0.316 | -0.201 ± 0.043 | <0.001 | 12.15 | 8.00 | 78 | 0.67 | 0.131 | 0.09 |
| greedy | 1.948 ± 0.225 | - | - | 12.29 | 7.13 | 56 | 0.59 | 0.147 | 0.24 |
| hill climbing | 1.474 ± 0.121 | +0.474 ± 0.047 | <0.001 | 7.08 | 6.74 | 71 | 0.77 | 0.007 | 1.32 |
| RL greedy dekod | 1.645 ± 0.140 | +0.303 ± 0.050 | <0.001 | 7.73 | 6.89 | 82 | 0.82 | 0.015 | 0.10 |
| RL sampling 32 | 1.501 ± 0.076 | +0.447 ± 0.051 | <0.001 | 6.87 | 6.55 | 77 | 0.83 | 0.005 | 3.04 |


## 3.2 Prenos na instance iz literature

Mandl i Mumford se puštaju sa svojim standardnim parametrima, dakle van
distribucije na kojoj je politika trenirana.

Radi na svih pet instanci i tražnju pokriva u celini, ali po cilju zaostaje za
lokalnom pretragom od 8% na Mandlu do 21% na Mumford1. Zaostatak ne prati veličinu:
na Mumford3 sa 127 čvorova i 60 linija je 15%, dakle manji nego na instanci sa 70
čvorova, a to je instanca preko četiri puta veća od najvećeg grada u treningu.

Uzrok nije kapacitet nego izoštrenost: politika je do kraja treninga pala na
entropiju 0.13, pa 32 uzorka daju varijacije jedne ideje, a kad ta ideja ne odgovara
instanci nema šta da se bira. Raznovrsnost se vraća samim dekodiranjem, pa se kroz
rad istih 32 uzorka raspoređuje preko temperatura 1, 2, 4 i 8 (`temps-transfer.md`).

In [4]:
tabela('bench-transfer')

# Transfer na benchmark instance (model runs/gravity-v2h/best.pt, alpha=0.5)

Politika je trenirana na sintetici sa n [15, 30], R=4, dužina linije [2, 8].
Instance se puštaju sa svojim standardnim parametrima, dakle van te
distribucije, kolona `van distr.` kaže koliko.

| instanca | n | R | metoda | cilj | C_p_all | C_p | C_o | d_un | s |
|---|---|---|---|---|---|---|---|---|---|
| Mandl1 | 15 | 6 | random 200 | 1.491 | 12.54 | 12.54 | 109 | 0.000 | 0.1 |
| Mandl1 | 15 | 6 | hill climbing | 1.241 | 14.19 | 14.19 | 67 | 0.000 | 1.2 |
| Mandl1 | 15 | 6 | RL greedy dekod | 1.741 | 11.66 | 11.66 | 146 | 0.000 | 0.1 |
| Mandl1 | 15 | 6 | RL sampling 32 | 1.334 | 12.56 | 12.56 | 89 | 0.000 | 3.3 |
| Mandl1 | 15 | 6 | greedy | 1.313 | 11.82 | 11.82 | 91 | 0.000 | 0.2 |
| Mumford0 | 30 | 12 | random 200 | 2.356 | 25.31 | 25.31 | 260 | 0.000 | 0.3 |
| Mumford0 | 30 | 12 | hill climbing | 1.559 | 24.24 | 24.24 | 118 | 0.000 | 3.8 |
| Mumford0 | 30 | 12 | RL greedy dekod | 2.338 | 23.19 | 23.19 | 272 | 0.000 | 0.3 |
| Mumford0 | 30 | 12 | RL sampling 32 | 1.853 | 24.70 | 24.70 | 170 | 0.000 | 7.2 |
| Mumford0 | 30 | 12 | greedy | 1.686 | 21.44 | 21.44 | 162 | 0.000 | 2.1 |
| Mumford1 | 70 | 15 | random 200 | 3.059 | 30.25 | 30.25 | 1037 | 0.000 | 1.1 |
| Mumford1 | 70 | 15 | hill climbing | 2.134 | 32.20 | 32.20 | 592 | 0.000 | 6.2 |
| Mumford1 | 70 | 15 | RL greedy dekod | 2.856 | 36.03 | 36.03 | 876 | 0.000 | 0.9 |
| Mumford1 | 70 | 15 | RL sampling 32 | 2.586 | 32.46 | 32.46 | 795 | 0.000 | 24.5 |
| Mumford2 | 110 | 56 | random 200 | 5.782 | 32.71 | 32.71 | 3571 | 0.000 | 4.2 |
| Mumford2 | 110 | 56 | hill climbing | 3.631 | 30.62 | 30.62 | 2082 | 0.000 | 50.6 |
| Mumford2 | 110 | 56 | RL greedy dekod | 6.475 | 43.48 | 43.48 | 3890 | 0.000 | 4.4 |
| Mumford2 | 110 | 56 | RL sampling 32 | 4.308 | 34.78 | 34.78 | 2495 | 0.000 | 97.8 |
| Mumford3 | 127 | 60 | random 200 | 6.284 | 34.76 | 34.76 | 4398 | 0.000 | 5.6 |
| Mumford3 | 127 | 60 | hill climbing | 4.081 | 37.11 | 37.11 | 2625 | 0.000 | 37.4 |
| Mumford3 | 127 | 60 | RL sampling 32 | 4.709 | 39.57 | 39.57 | 3081 | 0.000 | 121.1 |


## 3.3 Ablacije i varijansa po seedu

Zato što je rasipanje između semena veće od svakog izmerenog efekta. Četiri semena
osnovne konfiguracije daju 1.564, 1.572, 1.488 i 1.514, dakle 1.534 ± 0.040, a
nijedna ispitana izmena ne izlazi iz tog pojasa: mere centralnosti na ulazu 1.512,
fiksno `alpha = 0.5` 1.518, self-critical baseline 1.519, izostavljena
standardizacija advantage-a 1.531, entropijski bonus podignut na 0.03 odnosno 0.1
daje 1.523 i 1.490. Prve četiri su prosek dva semena, poslednje dve jedno.

Jedini hiperparametar koji pomera rezultat je stopa učenja: 1e-4 daje 1.673, a
najbolja konfiguracija, 3e-4 uz batch 32, daje 1.489.

Uparen Wilcoxon daje `p < 0.01` i kad se porede dva semena **iste** konfiguracije.
Značajnost tamo, dakle, meri razliku između dva treninga, a ne efekat izmene koja
se ispituje. Sirove tabele po varijanti su u `results/abl-*.md`, a prave se
skriptom `tndp.experiments.bench_synth` nad odgovarajućim checkpointom.

## 3.4 Novi Sad

**Kaže:** po ovom cilju i na ovom modelu tražnje, sve automatske metode nalaze
jeftiniju mrežu od postojeće.

**Ne kaže da je bolja.** GSP mreža nije projektovana po ovom cilju. Cilj ne vidi
frekvencije, kapacitet ni vozni park, a upravo to određuje postojeće linije. Trase
su svedene na proste puteve u zonskom grafu, dakle aproksimirane. I mreža koja
postoji nosi ograničenja koja model ne zna: infrastrukturu, okretnice, istorijske
odluke.

Zato je uz `cilj` bitniji `Jaccard`, koji meri poklapanje koridora i ne zavisi od
izbora cilja. Po njemu nijedna metoda ne rekonstruiše postojeće koridore naročito
verno, a politika najmanje.

In [5]:
tabela('novisad-poredjenje')

# Novi Sad: model naspram postojeće GSP mreže

Politika je trenirana na SINTETIČKIM gradovima (n [28, 36], R=19, dužina linije [2, 14])
i puštena na Novi Sad u jednom prolazu, bez dotreniravanja. To je tvrdnja
koju rad proverava.

Instanca: n=32 zona, R=19 linija, dužina [2, 14], alpha=0.5.
Model: `runs/novisad-r19h/best.pt`.

`cilj` je isti skalar za sve metode, manje je bolje. `Jaccard` meri
poklapanje sa GSP mrežom po parovima uzastopnih zona, koliko model
gradi iste koridore kao stvarni planeri.

| metoda | cilj | vs GSP | C_p_all | C_p | C_o | d_0 | d_un | Jaccard | s |
|---|---|---|---|---|---|---|---|---|---|
| GSP (postojeća) | 3.319 | - | 16.45 | 16.45 | 301.4 | 0.33 | 0.000 | 1.000 | 0.0 |
| random 200 | 3.148 | +0.171 | 16.11 | 16.11 | 282.4 | 0.29 | 0.000 | 0.549 | 0.4 |
| greedy | 1.682 | +1.637 | 16.27 | 16.27 | 99.0 | 0.18 | 0.000 | 0.452 | 4.0 |
| hill climbing | 1.649 | +1.670 | 16.20 | 16.20 | 95.3 | 0.20 | 0.000 | 0.435 | 5.9 |
| RL greedy dekod | 2.070 | +1.249 | 25.73 | 25.73 | 83.1 | 0.37 | 0.000 | 0.373 | 0.2 |
| RL sampling 32 | 1.769 | +1.551 | 20.80 | 20.80 | 79.0 | 0.29 | 0.000 | 0.446 | 6.3 |

`vs GSP` je razlika u cilju prema postojećoj mreži; pozitivno znači
bolje od GSP-a po ovom cilju. Treba ga čitati uz ograničenja niže.

## Šta ovo poređenje NE kaže

GSP mreža nije projektovana po ovom cilju, pa je porediti po njemu je
delom nepravedno u oba smera:

- cilj ne vidi frekvencije, kapacitet ni vozni park, a GSP linije postoje
  u režimu u kom te stvari odlučuju (vidi results/novisad-frekvencije.md)
- GSP trase su prevedene u zonski graf, pri čemu su svedene na proste
  puteve i popunjene po susedstvu (results koje daje tndp/novisad/instanca.py);
  to je aproksimacija stvarne trase
- mreža koja postoji nosi i ograničenja koja model ne zna: infrastrukturu,
  okretnice, kolektivne ugovore, istorijske odluke

Zato je `Jaccard` uz `cilj` bitniji nego sam `cilj`: on kaže da li model
prepoznaje iste koridore, što je tvrdnja koja ne zavisi od toga da li je
naša funkcija cilja ista kao ona koju je GSP imao na umu.



In [ ]:
Image(str(KOREN / 'results' / 'novisad-rl.png'))

## 3.5 Frekvencije

Model dimenzioniše intervale iz opterećenja, bez ijednog podatka o redu vožnje,
pa se poredi sa onim što GSP stvarno vozi.

Poredak je upotrebljiv jer model pogađa koja je linija jača: Spearmanova korelacija
sa objavljenim intervalima je +0.543, Pearsonova +0.672.

Pojedinačne vrednosti nisu, jer greška nije ravnomerna nego **dvopolna**: linija je
ili na podu od 5 minuta ili na plafonu od 60, pa je medijana greške 7 minuta. Šest
od devetnaest linija sedi na plafonu. Uzrok je što se interval izvodi iz jedne
najopterećenije deonice pa tvrdo odseca na dozvoljeni raspon. Ranije je bilo i gore:
dodela najkraćim putem je u koridoru sa paralelnim linijama davala sve pobedniku, pa
je petlja praznila devet od devetnaest linija. Opterećenje se sada deli po
frekvencijama među linijama koje nose istu vožnju, nijedna linija ne ostaje bez
putnika, ali dvopolnost ostaje. Popravka traži dodelu po strategiji
(Spiess-Florian), koja ovde nije implementirana.

In [7]:
tabela('novisad-frekvencije')

# Frekvencije: model naspram stvarnog reda vožnje

Frekvencijska faza (`core/frequencies.evaluate`) dimenzioniše intervale iz
opterećenja najopterećenije deonice. Do sad je puštana samo na Mandlu i
Mumfordu, gde nema reda vožnje pa nema ni provere. Ovde se pušta na
**postojeću GSP mrežu Novog Sada**, čiji je red vožnje poznat.

Stvarni interval je broj polazaka u vršnom satu (13:00-14:00,
radni dan, smer A). Model ne dobija nijedan podatak o redu vožnje,
izvodi intervale samo iz tražnje, trasa i kapaciteta vozila.

| linija | red vožnje | model | razlika | vozila |
|---|---|---|---|---|
| 7 | 7.5 min | 42.8 min | +35.3 min | 2 |
| 11 | 8.6 min | 5.0 min | -3.6 min | 10 |
| 3 | 10.0 min | 9.5 min | -0.5 min | 4 |
| 8 | 10.0 min | 5.0 min | -5.0 min | 9 |
| 1 | 12.0 min | 5.1 min | -6.9 min | 8 |
| 5 | 12.0 min | 20.9 min | +8.9 min | 3 |
| 9 | 12.0 min | 5.0 min | -7.0 min | 11 |
| 2 | 15.0 min | 5.0 min | -10.0 min | 6 |
| 4 | 15.0 min | 60.0 min | +45.0 min | 1 |
| 6 | 15.0 min | 5.0 min | -10.0 min | 10 |
| 12 | 15.0 min | 15.7 min | +0.7 min | 3 |
| 10 | 20.0 min | 57.2 min | +37.2 min | 1 |
| 13 | 20.0 min | 60.0 min | +40.0 min | 1 |
| 14 | 20.0 min | 5.0 min | -15.0 min | 11 |
| 15 | 20.0 min | 5.0 min | -15.0 min | 7 |
| 16 | 60.0 min | 60.0 min | +0.0 min | 1 |
| 17 | 60.0 min | 60.0 min | +0.0 min | 1 |
| 18 | 60.0 min | 60.0 min | +0.0 min | 1 |
| 19 | 60.0 min | 60.0 min | +0.0 min | 1 |

## Poklapanje

| mera | vrednost |
|---|---|
| medijana |greška| | +7.000 |
| prosečna greška | +4.953 |
| Pearson | +0.672 |
| Spearman | +0.543 |
| unutar 5 min | +0.421 |

Model traži **91 vozila** za celu mrežu, uz prosečno čekanje 3.3 min.

## Osetljivost na pretpostavke

Kapacitet vozila i udeo vršnog sata su pretpostavke, ne merenja, pa se
poklapanje meri i van podrazumevanih vrednosti.

| konstanta | vrednost | medijana \|greška\| | Spearman |
|---|---|---|---|
| kapacitet | 60 | 7.0 min | +0.565 |
| kapacitet | 80 | 7.0 min | +0.543 |
| kapacitet | 100 | 7.0 min | +0.434 |
| kapacitet | 120 | 7.0 min | +0.396 |
| udeo vrha | 0.08 | 7.0 min | +0.434 |
| udeo vrha | 0.1 | 7.0 min | +0.543 |
| udeo vrha | 0.12 | 7.0 min | +0.572 |

Poklapanje se jedva menja sa tim konstantama, i to je samo po sebi
dijagnoza: većina linija je prikovana za donju ili gornju granicu
intervala, pa ih pomeranje kapaciteta nema gde da pomeri.

## Gde model greši i zašto

Greška nije ravnomerna nego **dvopolna**: linija je ili na podu od 5 min
ili na plafonu od 60. Uzrok je što linija koja ostane bez putnika dobija
najređi dozvoljen interval.

| režim | linija bez ijednog putnika | od toga u brojanju 2017 | udeo prevoza koji nose |
|---|---|---|---|
| prvi prolaz, fiksni penal | 0 od 19 | 0 () | 0.0% |
| posle konvergencije petlje | 0 od 19 | 0 () | 0.0% |
| sa stvarnim redom vožnje | 0 od 19 | 0 () | 0.0% |

Nijedan režim ne isprazni ni jednu liniju. Ranije je čista dodela
najkraćim putem praznila dve, a petlja pogoršavala na devet: u koridoru
sa paralelnim linijama pobednik je uzimao sve, pa je linija bez
opterećenja dobijala dug interval i time postajala još manje privlačna.
Opterećenje se sada deli po frekvencijama među linijama koje nose istu
vožnju, pa te povratne sprege nema.

Brojevi pre te ispravke, na koje se rad poziva u kritičkom osvrtu
(regenerisati se ne mogu jer je uzrok uklonjen, vidi commit `c902ba7`):

| režim | linija bez putnika | od toga u brojanju 2017 | udeo prevoza |
|---|---|---|---|
| prvi prolaz, fiksni penal | 2 od 19 | 2 (4, 16) | 8.8% |
| posle konvergencije petlje | 9 od 19 | 8 (4, 5, 7, 10, 13, 16, 17, 18) | 30.0% |
| sa stvarnim redom vožnje | 6 od 19 | 5 (4, 13, 16, 17, 18) | 12.3% |

To je aproksimacija Spiess-Florianovog modela strategija, ne on sam: deli
se samo vožnja koju je najkraći put već izabrao, a optimalna strategija se
ne traži iznova. Funkcija cilja time nije dirnuta, jer u nju ulaze vremena
putovanja i vreme vožnje linija, a ne opterećenja po liniji.

Dvopolnost intervala time nije uklonjena. Interval se izvodi iz
najopterećenije deonice pa tvrdo odseca na dozvoljeni raspon, što veliki deo
linija gura na jednu od dve granice. Uz to je nivo usluge delom politička
odluka: GSP vozi liniju 7 na 7,5 minuta jer je tako odlučeno, a ne zato što
je tražnja to iznudila.

**Posledica za rad:** poredak linija po opterećenju je upotrebljiv
(Spearman +0.543), pojedinačni intervali nisu. Svaki
zaključak koji traži tačan interval po liniji, a tu spada i poređenje
vidova prevoza sa tramvajem, mora sačekati dodelu po strategiji.



## 3.6 Šta nije radilo

Pet stvari je provereno i nije se potvrdilo. Svaka je izmerena i ima objašnjenje.

| nalaz | gde |
|---|---|
| featuri iz analize kompleksnih mreža ne pomažu | `results/abl-akm-h.md` |
| MCTS dekoder ne tuče sampling | `results/bench-decoders.md` |
| frekvencijska faza se raspada na paralelnim linijama | `results/novisad-frekvencije.md` |
| beta se ne da kalibrisati, prag može | `results/novisad-kalibracija.md` |
| cilj na `alpha = 0.1` preferira mrežu bez veze za pola tražnje | `results/pareto.md` |

Zato što se može proveriti, ponoviti i pobiti.

"Betweenness ne pomaže" uz izmereno objašnjenje, a ono glasi: rang transformisan
betweenness vuče politiku u iste centralne čvorove, pa `C_p_all` padne ali `C_o`
poraste, jer projektovanje traži da se linije razmaknu a centralnost ih skuplja.
Takav nalaz kaže i gde bi taj feature mogao da radi, a gde ne. Tvrdnja bez provere
ne kaže ništa ni u jednom smeru, a tvrdnja bez mehanizma ne kaže šta dalje.

## Zaključci

Pokazano je da politika trenirana samo na sintetici u jednom prolazu drži korak sa
lokalnom pretragom: 1.501 naspram 1.474 na gradovima van treninga, uz 32 evaluacije
cilja naspram 3000. Pretpostavka je ostalo sve što traži tačno opterećenje po liniji,
jer raspoređivanje bira jedan najkraći put po paru zona.

Najjači nalaz je hibrid: lokalna pretraga koja kreće iz mreže politike daje 1.433,
najbolje u repou, uz `p < 0.001` na svih šest vrednosti alpha. Nije artefakt jer je
poređenje uparen test na istim gradovima i pod istim ukupnim budžetom od 3000
evaluacija, u kom se i cena starta naplaćuje; greedy start pod istim uslovima ne daje
ništa merljivo (1.503, `p = 0.312`).

Sledeći korak je zamena raspoređivanja modelom strategija (Spiess-Florian), jer na
njega naležu i kalibracija tražnje i izvođenje intervala sleđenja, pa se bez njega ni
dvopolnost frekvencija ne može ukloniti.